In [0]:
from pyspark.sql.functions import col,upper

In [0]:
df = spark.table("associate_assignment.default.bronze_customer_cdc")

In [0]:
df_valid_customer = (
    df
    .filter(
        (col("customer_id").rlike("^[0-9]+$")) &
        (col('name').isNotNull()) & (col('name') != " ") &
        (col("city").isNotNull() & upper(col("status")).isin(["ACTIVE", "INACTIVE", "PENDING", "DELETED"]) & upper(col("operation")).isin(["I", "U","D"])))
    .withColumn("customer_id", col("customer_id").try_cast("int"))
    .filter(col("customer_id") > 0)
)

In [0]:
%sql
drop table if exists associate_assignment.default.dim_customer_cdc

In [0]:
df_valid_customer.write.mode("overwrite").saveAsTable("associate_assignment.default.dim_customer_cdc")